# LeetCode #22 — Generate Parentheses: Backtracking có cắt tỉa

**Mức độ:** Medium · **Chủ đề:** String, Backtracking, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def generateParenthesis(self, n: int) -> List[str]:
```

Ở bài #17, backtracking sinh **mọi** tổ hợp. Bài này khác: phần lớn các chuỗi gồm `(` và `)` là **không hợp lệ**. Mục tiêu của notebook là học **cắt tỉa (pruning)**: chỉ đi vào nhánh nào **còn có thể** dẫn tới đáp án, nhờ đó không bao giờ sinh ra chuỗi sai.

---

## Sau notebook này, bạn sẽ nắm được

1. Thế nào là một chuỗi ngoặc **hợp lệ**, và cách kiểm tra bằng một **bộ đếm**.
2. Cách "sinh hết rồi lọc" và vì sao nó lãng phí.
3. Hai quy tắc cắt tỉa: `open < n` và `close < open`.
4. Mẫu backtracking với **trạng thái** (`open`, `close`) thay vì chỉ `index`.
5. Số lượng kết quả là **số Catalan**.

# 1. Bài toán là gì?

Cho `n` cặp ngoặc tròn. Sinh **mọi** chuỗi ngoặc **hợp lệ** (well-formed) dùng đúng `n` dấu `(` và `n` dấu `)`.

**Constraints (theo đề):** `1 <= n <= 8`

---

## Ví dụ

```text
n = 3  ->  ["((()))", "(()())", "(())()", "()(())", "()()()"]
n = 1  ->  ["()"]
```

Thứ tự kết quả tùy ý.

## Thế nào là hợp lệ?

- Mỗi `(` có một `)` đóng nó, **đóng sau khi mở**.
- `"(())"`, `"()()"` hợp lệ. `")("`, `"(()"`, `"())("` không hợp lệ.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Đệ quy (recursion)
2. Quay lui (backtracking)
3. Ngăn xếp (stack)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [1]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [2]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [3]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [4]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [5]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [6]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [7]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [8]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [9]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [10]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Ngăn xếp (stack)

### Ngăn xếp là gì?

Ngăn xếp là cấu trúc **vào sau, ra trước** (LIFO — Last In, First Out), giống chồng đĩa: chỉ đặt thêm hoặc lấy ra ở **đỉnh**.

```text
push 1   push 2   push 3   pop → 3   top = 2
                   │ 3 │
          │ 2 │    │ 2 │    │ 2 │
 │ 1 │    │ 1 │    │ 1 │    │ 1 │
 └───┘    └───┘    └───┘    └───┘
```

| Thao tác | Ý nghĩa | Python (`list`) | Độ phức tạp |
|---|---|---|---|
| `push(x)` | đặt `x` lên đỉnh | `st.append(x)` | `O(1)` |
| `pop()` | lấy và bỏ đỉnh | `st.pop()` | `O(1)` |
| `top()` | xem đỉnh | `st[-1]` | `O(1)` |
| rỗng? | | `not st` | `O(1)` |

In [11]:
st = []
for x in [1, 2, 3]:
    st.append(x)
    print("push", x, "->", st)
print("top:", st[-1])
while st:
    print("pop", st.pop(), "->", st)
try:
    st.pop()
except IndexError as e:
    print("pop khi rỗng:", e)

push 1 -> [1]
push 2 -> [1, 2]
push 3 -> [1, 2, 3]
top: 3
pop 3 -> [1, 2]
pop 2 -> [1]
pop 1 -> []
pop khi rỗng: pop from empty list


### Khi nào nghĩ tới ngăn xếp?

Khi thứ được mở **sau cùng** phải được đóng/xử lý **trước tiên**:

- Dấu ngoặc lồng nhau: `( [ { } ] )`.
- Đường dẫn thư mục: `..` hủy thư mục vừa vào.
- Tính biểu thức hậu tố, gọi hàm đệ quy (máy tính dùng *call stack*).
- "Hoàn tác" (undo).

Ví dụ kinh điển — kiểm tra ngoặc hợp lệ: gặp ngoặc mở thì push; gặp ngoặc đóng thì đỉnh phải là ngoặc mở tương ứng.

In [12]:
def ngoac_hop_le(s):
    cap = {")": "(", "]": "[", "}": "{"}
    st = []
    for ch in s:
        if ch in "([{":
            st.append(ch)
        else:
            if not st or st[-1] != cap[ch]:
                return False
            st.pop()
    return not st                        # còn ngoặc mở chưa đóng → sai

for s in ["([]{})", "([)]", "((", "}"]:
    print(f"{s!r:>9}: {ngoac_hop_le(s)}")

 '([]{})': True
   '([)]': False
     '((': False
      '}': False


### Lỗi hay gặp

- `pop()` / `st[-1]` khi ngăn xếp rỗng → `IndexError`. Luôn kiểm tra `if st`.
- Quên kiểm tra ngăn xếp **còn sót** phần tử ở cuối.
- Dùng `list.pop(0)` / `insert(0, x)` (đầu danh sách) — đó là hàng đợi và tốn `O(n)`; ngăn xếp luôn thao tác ở **cuối** list.

## 2.4 Áp dụng vào bài này

Sinh mọi chuỗi ngoặc hợp lệ bằng **quay lui**: ở mỗi bước chọn thêm `(` hoặc `)`, nhưng **cắt tỉa** — chỉ thêm `(` khi còn, chỉ thêm `)` khi số `)` đang ít hơn số `(`. Điều kiện "hợp lệ" chính là điều kiện của bài kiểm tra ngoặc bằng **ngăn xếp**.

# 3. Nền tảng: kiểm tra hợp lệ bằng bộ đếm

Đi từ trái sang phải, giữ `balance` = số `(` **chưa được đóng**:

- Gặp `(` → `balance += 1`.
- Gặp `)` → `balance -= 1`.
- Nếu có lúc `balance < 0` → đóng nhiều hơn mở → **sai ngay**.
- Cuối cùng phải có `balance == 0`.

(Với chỉ **một loại** ngoặc, bộ đếm là đủ; bài #20 Valid Parentheses có nhiều loại ngoặc nên cần stack.)

In [13]:
def is_valid(s: str) -> bool:
    balance = 0
    for c in s:
        balance += 1 if c == "(" else -1
        if balance < 0:
            return False
    return balance == 0

for s in ["(())", "()()", ")(", "(()", "())(", ""]:
    print(f"{s!r:8} -> {is_valid(s)}")

'(())'   -> True
'()()'   -> True
')('     -> False
'(()'    -> False
'())('   -> False
''       -> True


# 4. Cách 1: sinh mọi chuỗi rồi lọc

Có `2n` vị trí, mỗi vị trí chọn `(` hoặc `)` → `2^(2n)` chuỗi. Lọc ra chuỗi hợp lệ.

In [14]:
from typing import List
from itertools import product

def generate_bruteforce(n: int) -> List[str]:
    result = []
    for chars in product("()", repeat=2 * n):
        s = "".join(chars)
        if is_valid(s):
            result.append(s)
    return result

print(generate_bruteforce(3))
for n in range(1, 7):
    total = 2 ** (2 * n)
    valid = len(generate_bruteforce(n))
    print(f"n = {n}: sinh {total:>5} chuỗi, chỉ {valid:>3} hợp lệ ({100 * valid / total:.1f}%)")

['((()))', '(()())', '(())()', '()(())', '()()()']
n = 1: sinh     4 chuỗi, chỉ   1 hợp lệ (25.0%)
n = 2: sinh    16 chuỗi, chỉ   2 hợp lệ (12.5%)
n = 3: sinh    64 chuỗi, chỉ   5 hợp lệ (7.8%)
n = 4: sinh   256 chuỗi, chỉ  14 hợp lệ (5.5%)
n = 5: sinh  1024 chuỗi, chỉ  42 hợp lệ (4.1%)
n = 6: sinh  4096 chuỗi, chỉ 132 hợp lệ (3.2%)


**Lãng phí:** với `n = 6`, ta sinh `4096` chuỗi để giữ lại `132`. Với `n = 8` là `65536` chuỗi, mỗi chuỗi kiểm tra `O(n)`.

Tệ hơn, ta biết chuỗi `")..."` sai **ngay từ ký tự đầu**, nhưng vẫn sinh cả `2^(2n-1)` chuỗi bắt đầu bằng `)`.

> Ý tưởng: **kiểm tra trong lúc sinh**, dừng một nhánh ngay khi nó chắc chắn sai.

# 5. Hai quy tắc cắt tỉa

Đang xây dở một chuỗi, gọi `open` = số `(` đã đặt, `close` = số `)` đã đặt. Khi nào được đặt thêm?

1. **Đặt `(`** khi `open < n` — còn ngoặc mở để dùng.
2. **Đặt `)`** khi `close < open` — đang có ngoặc mở chưa được đóng.

Hai quy tắc này đảm bảo:

- `balance = open - close` **không bao giờ âm** (quy tắc 2).
- Không dùng quá `n` ngoặc mở (quy tắc 1), và `close <= open <= n`.

Khi `len(path) == 2n` (tức `open == close == n`), chuỗi **chắc chắn hợp lệ** — không cần gọi `is_valid`.

Ngược lại, **mọi** chuỗi hợp lệ đều tôn trọng hai quy tắc này ở mọi tiền tố, nên ta không bỏ sót.

In [15]:
n = 2
for prefix in ["", "(", "((", "()", "(()"]:
    o, c = prefix.count("("), prefix.count(")")
    can_open = o < n
    can_close = c < o
    print(f"{prefix!r:6} open={o} close={c} -> đặt '(' được? {can_open!s:5}  đặt ')' được? {can_close}")

''     open=0 close=0 -> đặt '(' được? True   đặt ')' được? False
'('    open=1 close=0 -> đặt '(' được? True   đặt ')' được? True
'(('   open=2 close=0 -> đặt '(' được? False  đặt ')' được? True
'()'   open=1 close=1 -> đặt '(' được? True   đặt ')' được? False
'(()'  open=2 close=1 -> đặt '(' được? False  đặt ')' được? True


# 6. Cách 2: Backtracking có cắt tỉa

```python
def backtrack(open, close):
    if len(path) == 2 * n:
        result.append("".join(path))
        return
    if open < n:
        path.append("(");  backtrack(open + 1, close);  path.pop()
    if close < open:
        path.append(")");  backtrack(open, close + 1);  path.pop()
```

In [16]:
def generate_parenthesis(n: int) -> List[str]:
    result, path = [], []

    def backtrack(open_count: int, close_count: int):
        if len(path) == 2 * n:
            result.append("".join(path))
            return
        if open_count < n:
            path.append("(")
            backtrack(open_count + 1, close_count)
            path.pop()
        if close_count < open_count:
            path.append(")")
            backtrack(open_count, close_count + 1)
            path.pop()

    backtrack(0, 0)
    return result

print(generate_parenthesis(3))
print(generate_parenthesis(1))

['((()))', '(()())', '(())()', '()(())', '()()()']
['()']


Tên biến `open_count` thay vì `open` để không che hàm có sẵn `open()` của Python.

# 7. Cây quyết định và mô phỏng

Với `n = 2`, cây chỉ còn các nhánh hợp lệ:

```text
                    ""
                    |
                   "("                 (không được đặt ')' khi open = close = 0)
                 /     \
             "(("       "()"
              |           |
            "(()"       "()("
              |           |
           "(())"      "()()"
```

Chỉ có **2 lá**, đúng bằng số kết quả. Không có nhánh nào bị bỏ phí.

In [17]:
def trace_generate(n: int) -> List[str]:
    result, path = [], []

    def backtrack(o: int, c: int):
        indent = "  " * len(path)
        if len(path) == 2 * n:
            print(f"{indent}LÁ: {''.join(path)!r}")
            result.append("".join(path))
            return
        if o < n:
            path.append("(")
            print(f"{indent}+ '(' -> {''.join(path)!r} (open={o + 1}, close={c})")
            backtrack(o + 1, c)
            path.pop()
        else:
            print(f"{indent}  (không đặt '(' nữa: open = n)")
        if c < o:
            path.append(")")
            print(f"{indent}+ ')' -> {''.join(path)!r} (open={o}, close={c + 1})")
            backtrack(o, c + 1)
            path.pop()

    backtrack(0, 0)
    return result

print(trace_generate(2))

+ '(' -> '(' (open=1, close=0)
  + '(' -> '((' (open=2, close=0)
      (không đặt '(' nữa: open = n)
    + ')' -> '(()' (open=2, close=1)
        (không đặt '(' nữa: open = n)
      + ')' -> '(())' (open=2, close=2)
        LÁ: '(())'
  + ')' -> '()' (open=1, close=1)
    + '(' -> '()(' (open=2, close=1)
        (không đặt '(' nữa: open = n)
      + ')' -> '()()' (open=2, close=2)
        LÁ: '()()'
['(())', '()()']


# 8. Phiên bản đúng format LeetCode

In [18]:
class Solution:
    def generateParenthesis(self, n: int) -> List[str]:
        result, path = [], []

        def backtrack(open_count: int, close_count: int):
            if len(path) == 2 * n:
                result.append("".join(path))
                return
            if open_count < n:
                path.append("(")
                backtrack(open_count + 1, close_count)
                path.pop()
            if close_count < open_count:
                path.append(")")
                backtrack(open_count, close_count + 1)
                path.pop()

        backtrack(0, 0)
        return result

In [19]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    assert len(actual) == len(set(actual)), (name, "có trùng")
    print(f"{name:<10} PASS  ({len(actual)} chuỗi) {actual[:5]}{' ...' if len(actual) > 5 else ''}")


sol = Solution()
check("n = 1", sol.generateParenthesis(1), ["()"])
check("n = 2", sol.generateParenthesis(2), ["(())", "()()"])
check("n = 3", sol.generateParenthesis(3), ["((()))", "(()())", "(())()", "()(())", "()()()"])
for n in range(1, 9):
    if n <= 6:
        assert sorted(sol.generateParenthesis(n)) == sorted(generate_bruteforce(n))
    assert all(is_valid(s) and len(s) == 2 * n for s in sol.generateParenthesis(n))
print("n = 1..8: mọi chuỗi đều hợp lệ; n = 1..6 khớp cách sinh-rồi-lọc")

n = 1      PASS  (1 chuỗi) ['()']
n = 2      PASS  (2 chuỗi) ['(())', '()()']
n = 3      PASS  (5 chuỗi) ['((()))', '(()())', '(())()', '()(())', '()()()']


n = 1..8: mọi chuỗi đều hợp lệ; n = 1..6 khớp cách sinh-rồi-lọc

# 9. Số lượng kết quả: số Catalan

In [20]:
from math import comb

for n in range(1, 9):
    catalan = comb(2 * n, n) // (n + 1)
    print(f"n = {n}: {len(sol.generateParenthesis(n)):>5} chuỗi, Catalan C_{n} = {catalan}")

n = 1:     1 chuỗi, Catalan C_1 = 1
n = 2:     2 chuỗi, Catalan C_2 = 2
n = 3:     5 chuỗi, Catalan C_3 = 5
n = 4:    14 chuỗi, Catalan C_4 = 14
n = 5:    42 chuỗi, Catalan C_5 = 42
n = 6:   132 chuỗi, Catalan C_6 = 132
n = 7:   429 chuỗi, Catalan C_7 = 429
n = 8:  1430 chuỗi, Catalan C_8 = 1430


Số chuỗi ngoặc hợp lệ với `n` cặp là **số Catalan** `C_n = C(2n, n) / (n + 1)`, tăng xấp xỉ `4^n / (n^1.5 · √π)`.

Dãy Catalan `1, 2, 5, 14, 42, 132, …` xuất hiện ở rất nhiều bài: số cây nhị phân tìm kiếm khác nhau (**#96**), số cách chia đa giác thành tam giác, …

# 10. Vì sao đúng? Độ phức tạp?

**Đúng (không sinh chuỗi sai):** quy tắc 2 giữ `balance >= 0` ở mọi tiền tố; khi dừng, `open = close = n` nên `balance = 0`. Đó đúng là định nghĩa hợp lệ (mục 3).

**Đúng (không bỏ sót):** mọi chuỗi hợp lệ, xét từng tiền tố, luôn có `open <= n` và `close <= open` — tức mọi ký tự của nó đều là một lựa chọn được phép. Vậy đường đi tương ứng có trong cây.

**Không trùng:** mỗi đường đi từ gốc tới lá ứng với đúng một chuỗi.

**Độ phức tạp:** số lá là `C_n`, mỗi lá tốn `O(n)` để tạo chuỗi. Số nút trong cây cũng bị chặn bởi `O(n · C_n)` (mỗi nút nằm trên đường tới ít nhất một lá).

```text
Time:  O(n · C_n) ≈ O(4^n / √n)
Space: O(n) cho path và độ sâu đệ quy (không tính kết quả)
```

So với sinh-rồi-lọc: `O(n · 4^n)` — chậm hơn khoảng `n^1.5` lần.

# 11. Những lỗi dễ mắc

## Lỗi 1: Điều kiện đặt `)` là `close < n`

Thiếu ràng buộc với `open` → sinh ra `")("`, `"())("`, …

In [21]:
def wrong_close_condition(n: int) -> List[str]:
    result, path = [], []

    def backtrack(o, c):
        if len(path) == 2 * n:
            result.append("".join(path))
            return
        if o < n:
            path.append("("); backtrack(o + 1, c); path.pop()
        if c < n:                                   # SAI: phải là c < o
            path.append(")"); backtrack(o, c + 1); path.pop()

    backtrack(0, 0)
    return result

bad = wrong_close_condition(2)
print("Sai :", bad, "-> hợp lệ?", [is_valid(s) for s in bad])

Sai : ['(())', '()()', '())(', ')(()', ')()(', '))(('] -> hợp lệ? [True, True, False, False, False, False]


## Lỗi 2: Dùng `elif` thay vì hai `if` độc lập

Hai lựa chọn `(` và `)` **đều** cần được thử. Nếu viết `elif`, khi được đặt `(` thì không bao giờ thử `)` → chỉ ra `"((...))"`.

In [22]:
def wrong_elif(n: int) -> List[str]:
    result, path = [], []

    def backtrack(o, c):
        if len(path) == 2 * n:
            result.append("".join(path))
            return
        if o < n:
            path.append("("); backtrack(o + 1, c); path.pop()
        elif c < o:                                 # SAI: elif
            path.append(")"); backtrack(o, c + 1); path.pop()

    backtrack(0, 0)
    return result

print("Sai :", wrong_elif(3))
print("Đúng:", generate_parenthesis(3))

Sai : ['((()))']
Đúng: ['((()))', '(()())', '(())()', '()(())', '()()()']


## Lỗi 3: Nối chuỗi rồi quên rằng mình đang sửa biến dùng chung

Cách viết khác hay gặp: truyền chuỗi hiện tại làm tham số `backtrack(cur + "(", ...)`. Cách này **không cần pop** vì mỗi lời gọi nhận một chuỗi mới (chuỗi là bất biến). Nhưng nếu trộn hai phong cách — dùng `path` list mà quên `pop` — sẽ sai như ở bài #17.

In [23]:
def generate_with_string_param(n: int) -> List[str]:
    result = []

    def backtrack(cur: str, o: int, c: int):
        if len(cur) == 2 * n:
            result.append(cur)
            return
        if o < n:
            backtrack(cur + "(", o + 1, c)
        if c < o:
            backtrack(cur + ")", o, c + 1)

    backtrack("", 0, 0)
    return result

print(generate_with_string_param(3))

['((()))', '(()())', '(())()', '()(())', '()()()']


Phong cách "truyền chuỗi" ngắn hơn nhưng tạo chuỗi mới ở mỗi nút (`O(n)` mỗi lần). Với `n ≤ 8`, cả hai đều rất nhanh.

# 12. Mẹo nhận diện pattern

**Backtracking có cắt tỉa** khi:

- Cần sinh **mọi** cấu hình thỏa ràng buộc.
- Có thể kiểm tra **trên tiền tố** (chuỗi đang xây dở) rằng nó **còn hy vọng** hay không.

Hãy tự hỏi: *"Trạng thái tối thiểu cần nhớ là gì?"* — ở đây chỉ là hai số `open`, `close`, không cần lưu cả chuỗi để kiểm tra.

## Template

```python
def backtrack(state):
    if <hoàn thành>:
        ghi kết quả; return
    for choice in lựa chọn:
        if <choice vi phạm ràng buộc trên tiền tố>:
            continue                  # CẮT TỈA
        áp dụng choice; backtrack(state mới); hoàn tác choice
```

Bài liên quan: **#20 Valid Parentheses**, **#32 Longest Valid Parentheses** (Hard), **#301 Remove Invalid Parentheses** (Hard), **#96 Unique Binary Search Trees** (Catalan).

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Vẽ cây quyết định cho `n = 3` (chỉ các nhánh hợp lệ). Đếm số nút ở mỗi tầng `0..6`.

## Bài 2 — Đoán output

```python
print(is_valid("(()())"), is_valid("())()("), is_valid("(((")))
print(len(generate_parenthesis(4)))
print(comb(10, 5) // 6)
```

## Bài 3 — Điền chỗ trống

In [24]:
def exercise_3_generate(n: int) -> List[str]:
    result, path = [], []

    def backtrack(open_count: int, close_count: int):
        # TODO 1: base case khi đủ 2n ký tự
        # TODO 2: nếu còn được đặt '(' -> đặt, đệ quy, gỡ
        # TODO 3: nếu được đặt ')' -> đặt, đệ quy, gỡ
        pass

    backtrack(0, 0)
    return result


# assert sorted(exercise_3_generate(3)) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
# assert exercise_3_generate(1) == ["()"]

## Bài 4 — Biến thể: độ sâu lồng tối đa

Sinh mọi chuỗi ngoặc hợp lệ với `n` cặp **có độ sâu lồng nhau không quá `d`** (độ sâu = giá trị lớn nhất của `balance`).

```python
generate_max_depth(3, 1) -> ["()()()"]
generate_max_depth(3, 2) -> ["(()())", "(())()", "()(())", "()()()"]
```

Hãy cắt tỉa, không sinh hết rồi lọc.

In [25]:
def exercise_4_generate_max_depth(n: int, d: int) -> List[str]:
    # TODO
    pass


# assert exercise_4_generate_max_depth(3, 1) == ["()()()"]
# assert sorted(exercise_4_generate_max_depth(3, 2)) == ["(()())", "(())()", "()(())", "()()()"]
# assert sorted(exercise_4_generate_max_depth(3, 3)) == sorted(generate_parenthesis(3))

## Bài 5 — Mini challenge: đếm bằng Quy hoạch động

Đếm số chuỗi ngoặc hợp lệ với `n` cặp **không sinh ra chúng**, bằng DP:

`dp[o][c]` = số cách hoàn thành chuỗi khi đã đặt `o` ngoặc mở và `c` ngoặc đóng.

```python
count_valid(3) -> 5
count_valid(8) -> 1430
count_valid(30) -> 3814986502092304
```

In [26]:
def exercise_5_count_valid(n: int) -> int:
    # TODO
    pass


# assert exercise_5_count_valid(3) == 5
# assert exercise_5_count_valid(8) == 1430
# assert exercise_5_count_valid(30) == 3814986502092304

# 14. Gợi ý

**Bài 1**
- Mức 1: tầng 0 có 1 nút (`""`), tầng 1 có 1 nút (`"("`).
- Mức 2: tầng cuối (tầng 6) có 5 nút = số kết quả.

**Bài 3**
- Mức 1: so với code mục 6.
- Mức 2: hai `if` độc lập, không `elif`.

**Bài 4**
- Mức 1: độ sâu hiện tại là `open - close`.
- Mức 2: đặt `(` chỉ khi `open < n` **và** `open - close < d`.

**Bài 5**
- Mức 1: `dp[n][n] = 1` (đã xong, có 1 cách: không làm gì).
- Mức 2: `dp[o][c] = (dp[o+1][c] nếu o < n) + (dp[o][c+1] nếu c < o)`.
- Mức 3: tính `o` từ `n` về `0`, `c` từ `o` về `0`; đáp án `dp[0][0]`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [27]:
def count_nodes_per_level(n: int):
    levels = [0] * (2 * n + 1)

    def backtrack(o, c):
        levels[o + c] += 1
        if o + c == 2 * n:
            return
        if o < n:
            backtrack(o + 1, c)
        if c < o:
            backtrack(o, c + 1)

    backtrack(0, 0)
    return levels

print(count_nodes_per_level(3))

[1, 1, 2, 3, 5, 5, 5]


## Đáp án Bài 2

In [28]:
print(is_valid("(()())"), is_valid("())()("), is_valid("((("))   # True False False
print(len(generate_parenthesis(4)))                                # 14
print(comb(10, 5) // 6)                                            # 42 = C_5

True False False
14
42


## Đáp án Bài 3

In [29]:
def answer_3_generate(n: int) -> List[str]:
    result, path = [], []

    def backtrack(open_count: int, close_count: int):
        if len(path) == 2 * n:
            result.append("".join(path))
            return
        if open_count < n:
            path.append("(")
            backtrack(open_count + 1, close_count)
            path.pop()
        if close_count < open_count:
            path.append(")")
            backtrack(open_count, close_count + 1)
            path.pop()

    backtrack(0, 0)
    return result

assert sorted(answer_3_generate(3)) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert answer_3_generate(1) == ["()"]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [30]:
def answer_4_generate_max_depth(n: int, d: int) -> List[str]:
    result, path = [], []

    def backtrack(o: int, c: int):
        if len(path) == 2 * n:
            result.append("".join(path))
            return
        if o < n and o - c < d:
            path.append("(")
            backtrack(o + 1, c)
            path.pop()
        if c < o:
            path.append(")")
            backtrack(o, c + 1)
            path.pop()

    backtrack(0, 0)
    return result

assert answer_4_generate_max_depth(3, 1) == ["()()()"]
assert sorted(answer_4_generate_max_depth(3, 2)) == ["(()())", "(())()", "()(())", "()()()"]
assert sorted(answer_4_generate_max_depth(3, 3)) == sorted(generate_parenthesis(3))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [31]:
def answer_5_count_valid(n: int) -> int:
    dp = [[0] * (n + 1) for _ in range(n + 1)]
    dp[n][n] = 1
    for o in range(n, -1, -1):
        for c in range(o, -1, -1):
            if o == n and c == n:
                continue
            ways = 0
            if o < n:
                ways += dp[o + 1][c]
            if c < o:
                ways += dp[o][c + 1]
            dp[o][c] = ways
    return dp[0][0]

assert answer_5_count_valid(3) == 5
assert answer_5_count_valid(8) == 1430
assert answer_5_count_valid(30) == 3814986502092304
assert all(answer_5_count_valid(n) == comb(2 * n, n) // (n + 1) for n in range(1, 40))
print("Bài 5 PASS (khớp công thức Catalan với n = 1..39)")

Bài 5 PASS (khớp công thức Catalan với n = 1..39)


DP chỉ tốn `O(n²)`, trong khi sinh hết tốn `O(n · C_n)` — với `n = 30` là không thể sinh nhưng đếm thì tức thì. Bài học: **đếm** thường dễ hơn nhiều so với **liệt kê**.

# 16. Tổng kết

> Backtracking + **cắt tỉa trên tiền tố**: chỉ đặt `(` khi `open < n`, chỉ đặt `)` khi `close < open`. Mọi lá đều là đáp án, không lãng phí.

---

## Checklist trước khi nộp

- [ ] Điều kiện đặt `(`: `open < n`?
- [ ] Điều kiện đặt `)`: `close < open` (không phải `close < n`)?
- [ ] Hai `if` độc lập, không `elif`?
- [ ] Base case `len(path) == 2 * n`?
- [ ] Có `pop` sau mỗi lần đệ quy (nếu dùng list)?